In [1]:
%load_ext dotenv
%dotenv

In [2]:
import os
import pygsheets
import pandas as pd
from openai import OpenAI
from pathlib import Path
from tqdm import tqdm

In [3]:
client = pygsheets.authorize()

In [4]:
data_file_key = os.environ['FILE_KEY']
sh = client.open_by_key(data_file_key)
wks1 = sh.worksheet('title', '다인')
wks2 = sh.worksheet('title', '진혁')
wks3 = sh.worksheet('title', '가휘')

In [5]:
questions_by_id = {}
for wks_id, wks in zip(['다인', '진혁', '가휘'], [wks1, wks2, wks3]):
    questions_by_id[wks_id] = wks.get_col(2)[1:]

In [6]:
len(questions_by_id['다인']), len(questions_by_id['진혁']), len(questions_by_id['가휘']) 

(350, 350, 400)

In [7]:
questions_by_id['다인'][:2]

['Temukan kota terpadat di Korea Selatan.',
 'Tolong sebutkan makanan liburan tradisional di Korea.']

### IndoGPT

In [8]:
import sys
sys.path.append('/home/kikiputri/ner-training/')

In [9]:
from transformers import GPT2LMHeadModel
from indobenchmark import IndoNLGTokenizer
import torch

[2023-11-09 14:55:13,299] [INFO] [real_accelerator.py:158:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [10]:
gpt_model = GPT2LMHeadModel.from_pretrained('indobenchmark/indogpt')
gpt_model = gpt_model.cuda()
tokenizer = IndoNLGTokenizer.from_pretrained('indobenchmark/indogpt')

In [15]:
def get_indogpt_response(question):
    str_input = f'<s> {question}'
    gpt_input = torch.LongTensor([tokenizer.encode(str_input, add_special_tokens=False)]).cuda()
    gpt_out = gpt_model.generate(
        gpt_input,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id, min_length=len(str_input)+1, max_length=len(str_input)+200
    )
    str_out = tokenizer.decode(gpt_out[0], skip_special_tokens=False)
    str_out = str_out.replace(str_input.lower(), '')
    str_out = str_out.replace('</s>', '').strip()

    return str_out

In [17]:
answer_by_id = {}
for id, questions in questions_by_id.items():
    answers = []
    for question in tqdm(questions, desc=f"Processing {id}"):
        answers.append(get_indogpt_response(question))
    answer_by_id[id] = answers

Processing 가휘: 100%|██████████| 400/400 [04:19<00:00,  1.54it/s]


In [18]:
for wks_id, wks in zip(['다인', '진혁', '가휘'], [wks1, wks2, wks3]):
    wks.update_col(6, ['IndoGPT'] + answer_by_id[wks_id])

### BLOOMZ

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [9]:
model_name = 'bigscience/bloomz-7b1'
model = AutoModelForCausalLM.from_pretrained(model_name, resume_download=True, device_map="cuda:0", torch_dtype=torch.float16)
model = model.cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name, truncation_side='left')
tokenizer.padding_side = 'left'

[2023-11-09 18:38:03,280] [INFO] [real_accelerator.py:158:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [19]:
def get_bloomz_response(question):
    str_input = f"{question}"
    input_size = len(str_input)
    input = tokenizer(str_input, return_tensors="pt").to('cuda')
    output = model.generate(**input, do_sample=True, min_length=input_size+1, max_length=input_size+200)
    preds = tokenizer.decode(output[0][input["input_ids"].shape[1]:], skip_special_tokens=True)
    return preds.strip()

In [20]:
answer_by_id = {}
for id, questions in questions_by_id.items():
    answers = []
    for question in tqdm(questions, desc=f"Processing {id}"):
        answers.append(get_bloomz_response(question))
    answer_by_id[id] = answers

Processing 가휘: 100%|██████████| 400/400 [08:43<00:00,  1.31s/it]


In [21]:
for wks_id, wks in zip(['다인', '진혁', '가휘'], [wks1, wks2, wks3]):
    wks.update_col(8, ['BLOOMZ (7b)'] + answer_by_id[wks_id])

### mT0

In [8]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

In [9]:
model_name = "bigscience/mt0-xxl"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="cuda:0", torch_dtype=torch.float16, resume_download=True)
model = model.cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name, truncation_side='left')

[2023-11-09 19:36:25,636] [INFO] [real_accelerator.py:158:get_accelerator] Setting ds_accelerator to cuda (auto detect)


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

/home/kikiputri/miniconda3/envs/ner/lib/python3.8/site-packages/huggingface_hub/file_download.py:133: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in /mnt/nas2/kikiputri/cache/hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
  warnings.warn(message)


In [10]:
def get_mt0_response(question):
    input = tokenizer(question, return_tensors="pt").to('cuda')
    output = model.generate(**input, do_sample=True, min_length=1, max_length=200)
    preds = tokenizer.decode(output[0], skip_special_tokens=True)
    return preds.strip()

In [12]:
answer_by_id = {}
for id, questions in questions_by_id.items():
    answers = []
    for question in tqdm(questions, desc=f"Processing {id}"):
        answers.append(get_mt0_response(question))
    answer_by_id[id] = answers

Processing 가휘: 100%|██████████| 400/400 [03:25<00:00,  1.95it/s]


In [13]:
for wks_id, wks in zip(['다인', '진혁', '가휘'], [wks1, wks2, wks3]):
    wks.update_col(9, ['mT0 (xxl)'] + answer_by_id[wks_id])

### LLaMA-2

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [9]:
model_name = "meta-llama/Llama-2-13b-chat-hf"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cuda:0", torch_dtype=torch.float16, resume_download=True)
model = model.bfloat16().cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name, truncation_side='left')
tokenizer.pad_token = tokenizer.bos_token
tokenizer.padding_side = "left"

[2023-11-09 20:31:46,229] [INFO] [real_accelerator.py:158:get_accelerator] Setting ds_accelerator to cuda (auto detect)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
def get_llama_response(question):
    input_size = len(question)
    input = tokenizer(question, return_tensors="pt").to('cuda')
    output = model.generate(**input, do_sample=True, min_length=input_size+1, max_length=input_size+200)
    preds = tokenizer.decode(output[0][input["input_ids"].shape[1]:], skip_special_tokens=True)
    return preds.strip()

In [12]:
answer_by_id = {}
for id, questions in questions_by_id.items():
    answers = []
    for question in tqdm(questions, desc=f"Processing {id}"):
        answers.append(get_llama_response(question))
    answer_by_id[id] = answers

Processing 가휘: 100%|██████████| 400/400 [1:07:06<00:00, 10.07s/it]


In [13]:
for wks_id, wks in zip(['다인', '진혁', '가휘'], [wks1, wks2, wks3]):
    wks.update_col(10, ['LLaMA-2 (13b)'] + answer_by_id[wks_id])

### ChatGPT (GPT-3.5)

In [8]:
oai_client = OpenAI(
    api_key=os.environ['OPENAI_API_KEY'],
    organization=os.environ['OPENAI_UILAB_KEY']
)

In [9]:
model_name = "gpt-3.5-turbo"
resp_history_filename = f"history/{model_name}_history_231109.csv"
resp_history_path = Path(resp_history_filename)
if resp_history_path.is_file():
    print("Response history found!")
    resp_history_df = pd.read_csv(resp_history_filename)
    response_history = dict(zip(resp_history_df.prompt, resp_history_df.response))
else:
    print("Response history not found. Initializing new one...")
    response_history = {}

Response history found!


In [10]:
def get_openai_response(input_prompt, model_name):
    if input_prompt in response_history:
        return response_history[input_prompt]
    
    resp = oai_client.chat.completions.create(
        model=model_name,
        messages=[
            {
                'role': 'user',
                'content': input_prompt
            }
        ],
        temperature=0.2,
    )
    resp_clean = resp.choices[0].message.content.strip()
    
    response_history[input_prompt] = resp_clean
    resp_history_df = pd.DataFrame({'prompt': response_history.keys(), 'response': response_history.values()})
    resp_history_df.to_csv(resp_history_filename, index=False)

    return resp_clean

In [11]:
answer_by_id = {}
for id, questions in questions_by_id.items():
    answers = []
    for question in tqdm(questions, desc=f"Processing {id}"):
        answers.append(get_openai_response(question, model_name))
    answer_by_id[id] = answers

Processing 가휘: 100%|██████████| 400/400 [00:11<00:00, 34.11it/s] 


In [12]:
for wks_id, wks in zip(['다인', '진혁', '가휘'], [wks1, wks2, wks3]):
    wks.update_col(7, ['ChatGPT (GPT-3.5)'] + answer_by_id[wks_id])